In [ ]:
#rds预处理
library(Seurat)
library(Matrix)
library(data.table)
library(CellEnergy)

# =========================================================
# 0. Input and output paths
# =========================================================

counts_path <- "path/GSE179994/GSE179994_all.Tcell.rawCounts.rds"
meta_path   <- "path/GSE179994/GSE179994_Tcell.metadata.tsv"

out_dir <- "path/GSE179994/Seurat_CellEnergy_full_pipeline"
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

cat("[INFO] Output directory:\n")
cat(out_dir, "\n\n")


Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Loading required package: marray

Loading required package: limma

Loading required package: mclust

Package 'mclust' version 6.1.2
Type 'citation("mclust")' for citing this R package in publications.


Attaching package: ‘mclust’


The following object is masked from ‘package:limma’:

    logsumexp


Loading required package: philentropy

Loading required package: plot3D

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: reticulate

Loading required package: parallel

Loading required package: plyr


Attaching package: ‘plyr’


The following object is masked from ‘package:mclust’:

    count


Loading required package: dplyr


Attaching package: ‘dplyr’


The following objects are masked from ‘package:plyr’:

    arrange, count, desc, mutate, rename, summarise, summariz

[INFO] Output directory:
/home/zhanghang/GSE179994/Seurat_CellEnergy_full_pipeline 



In [2]:
# =========================================================
# 1. Helper functions
# =========================================================

read_counts_robust <- function(path) {
  if (!file.exists(path)) {
    stop("[ERROR] Count file does not exist: ", path)
  }
  
  cat("[INFO] Count file information:\n")
  print(file.info(path))
  
  obj <- tryCatch(
    readRDS(path),
    error = function(e) {
      cat("[WARN] readRDS failed:\n")
      cat(conditionMessage(e), "\n")
      return(NULL)
    }
  )
  
  if (!is.null(obj)) {
    cat("[OK] Successfully read file using readRDS.\n")
    return(obj)
  }
  
  cat("[INFO] Trying load as RData/Rda...\n")
  
  env <- new.env()
  loaded_names <- tryCatch(
    load(path, envir = env),
    error = function(e) {
      cat("[WARN] load failed:\n")
      cat(conditionMessage(e), "\n")
      return(NULL)
    }
  )
  
  if (is.null(loaded_names)) {
    stop(
      "[ERROR] Cannot read this file as RDS or RData. ",
      "The file may be corrupted, incomplete, or not an R object."
    )
  }
  
  cat("[OK] Loaded object names:\n")
  print(loaded_names)
  
  if (length(loaded_names) == 1) {
    return(env[[loaded_names[1]]])
  }
  
  for (nm in loaded_names) {
    x <- env[[nm]]
    if (inherits(x, "dgCMatrix") || is.matrix(x) || is.data.frame(x)) {
      cat("[INFO] Using matrix-like object:\n")
      print(nm)
      return(x)
    }
  }
  
  stop("[ERROR] RData loaded, but no matrix-like object was found.")
}


extract_counts_matrix <- function(obj) {
  if (inherits(obj, "Seurat")) {
    cat("[INFO] Input object is Seurat. Extracting RNA counts...\n")
    
    counts <- tryCatch(
      GetAssayData(obj, assay = "RNA", layer = "counts"),
      error = function(e) {
        GetAssayData(obj, assay = "RNA", slot = "counts")
      }
    )
    
    return(counts)
  }
  
  if (inherits(obj, "dgCMatrix") || is.matrix(obj)) {
    cat("[INFO] Input object is matrix-like.\n")
    return(obj)
  }
  
  if (is.data.frame(obj)) {
    cat("[INFO] Input object is data.frame. Converting to matrix.\n")
    return(as.matrix(obj))
  }
  
  if (is.list(obj)) {
    cat("[INFO] Input object is list. Searching count matrix...\n")
    
    candidate_names <- c(
      "counts", "count", "rawCounts", "raw_counts",
      "expr", "expression", "matrix", "data"
    )
    
    found_name <- intersect(candidate_names, names(obj))[1]
    
    if (is.na(found_name)) {
      stop(
        "[ERROR] Cannot find count matrix in list. Available names:\n",
        paste(names(obj), collapse = ", ")
      )
    }
    
    cat("[INFO] Using list element:\n")
    print(found_name)
    
    return(obj[[found_name]])
  }
  
  stop("[ERROR] Unsupported count object type.")
}


to_sparse_matrix <- function(x) {
  if (inherits(x, "dgCMatrix")) {
    return(x)
  } else {
    return(as(as.matrix(x), "dgCMatrix"))
  }
}


write_matrix_with_id <- function(mat, file, id_col = "ID") {
  df <- as.data.frame(as.matrix(mat), check.names = FALSE)
  df <- cbind(
    setNames(data.frame(rownames(df), check.names = FALSE), id_col),
    df
  )
  fwrite(df, file = file)
}


minmax_scale_columns <- function(mat) {
  mat <- as.matrix(mat)
  
  col_min <- apply(mat, 2, min, na.rm = TRUE)
  col_max <- apply(mat, 2, max, na.rm = TRUE)
  col_range <- col_max - col_min
  
  col_range[col_range == 0] <- 1
  
  scaled <- sweep(mat, 2, col_min, FUN = "-")
  scaled <- sweep(scaled, 2, col_range, FUN = "/")
  
  return(scaled)
}


get_assay_data_compat <- function(obj, assay = "RNA", layer_name = "counts", slot_name = "counts") {
  DefaultAssay(obj) <- assay
  
  out <- tryCatch(
    GetAssayData(obj, assay = assay, layer = layer_name),
    error = function(e) {
      GetAssayData(obj, assay = assay, slot = slot_name)
    }
  )
  
  return(out)
}



In [3]:
# =========================================================
# 2. Read raw count matrix
# =========================================================

raw_obj <- read_counts_robust(counts_path)
counts <- extract_counts_matrix(raw_obj)
counts <- to_sparse_matrix(counts)

cat("[INFO] Raw count matrix dimension before orientation check:\n")
print(dim(counts))

cat("[INFO] Example row names:\n")
print(head(rownames(counts)))

cat("[INFO] Example column names:\n")
print(head(colnames(counts)))


# =========================================================
# 3. Read metadata
# =========================================================

metadata <- fread(
  meta_path,
  sep = "\t",
  data.table = FALSE,
  check.names = FALSE
)

cat("[INFO] Metadata dimension:\n")
print(dim(metadata))

cat("[INFO] Metadata columns:\n")
print(colnames(metadata))

# 默认 metadata 第一列是细胞 ID
cell_id_col <- colnames(metadata)[1]

metadata[[cell_id_col]] <- as.character(metadata[[cell_id_col]])
rownames(metadata) <- metadata[[cell_id_col]]

cat("[INFO] Cell ID column used:\n")
print(cell_id_col)

cat("[INFO] Example metadata cell IDs:\n")
print(head(rownames(metadata)))



[INFO] Count file information:
                                                                  size isdir
/home/zhanghang/GSE179994/GSE179994_all.Tcell.rawCounts.rds 2466366344 FALSE
                                                            mode
/home/zhanghang/GSE179994/GSE179994_all.Tcell.rawCounts.rds  664
                                                                          mtime
/home/zhanghang/GSE179994/GSE179994_all.Tcell.rawCounts.rds 2026-06-10 19:07:49
                                                                          ctime
/home/zhanghang/GSE179994/GSE179994_all.Tcell.rawCounts.rds 2026-06-10 19:07:49
                                                                          atime
/home/zhanghang/GSE179994/GSE179994_all.Tcell.rawCounts.rds 2026-06-10 19:08:03
                                                             uid  gid     uname
/home/zhanghang/GSE179994/GSE179994_all.Tcell.rawCounts.rds 1000 1000 zhanghang
                                             

In [4]:
# =========================================================
# 3. Read metadata
# =========================================================

metadata <- fread(
  meta_path,
  sep = "\t",
  data.table = FALSE,
  check.names = FALSE
)

cat("[INFO] Metadata dimension:\n")
print(dim(metadata))

cat("[INFO] Metadata columns:\n")
print(colnames(metadata))

# 默认 metadata 第一列是细胞 ID
cell_id_col <- colnames(metadata)[1]

metadata[[cell_id_col]] <- as.character(metadata[[cell_id_col]])
rownames(metadata) <- metadata[[cell_id_col]]

cat("[INFO] Cell ID column used:\n")
print(cell_id_col)

cat("[INFO] Example metadata cell IDs:\n")
print(head(rownames(metadata)))


[INFO] Metadata dimension:
[1] 150849      5
[INFO] Metadata columns:
[1] "cellid"   "patient"  "sample"   "celltype" "cluster" 
[INFO] Cell ID column used:
[1] "cellid"
[INFO] Example metadata cell IDs:
[1] "P1.ut.AAACCTGAGGCACATG-1" "P1.ut.AAACCTGGTGGTACAG-1"
[3] "P1.ut.AAACCTGTCACGGTTA-1" "P1.ut.AAACCTGTCCGCTGTT-1"
[5] "P1.ut.AAACCTGTCTGGTATG-1" "P1.ut.AAACCTGTCTTGAGAC-1"


In [5]:
# =========================================================
# 4. Select cell label column
# =========================================================
# 如果自动识别错了，手动改这里：
# label_col <- "你的细胞标签列名"

candidate_label_cols <- c(
  "celltype",
  "cell_type",
  "CellType",
  "cell.type",
  "annotation",
  "Annotation",
  "label",
  "Label",
  "cluster",
  "Cluster",
  "seurat_clusters"
)

label_col <- intersect(candidate_label_cols, colnames(metadata))[1]

if (is.na(label_col)) {
  cat("[ERROR] Cannot automatically find cell label column.\n")
  cat("[INFO] Available metadata columns:\n")
  print(colnames(metadata))
  stop("Please manually set label_col.")
}

cat("[INFO] Cell label column used:\n")
print(label_col)


[INFO] Cell label column used:
[1] "celltype"


In [6]:
# =========================================================
# 5. Make sure counts matrix is genes x cells
# =========================================================

meta_cells <- rownames(metadata)

overlap_cols <- sum(colnames(counts) %in% meta_cells)
overlap_rows <- sum(rownames(counts) %in% meta_cells)

cat("[INFO] Overlap with metadata cells in count matrix columns:\n")
print(overlap_cols)

cat("[INFO] Overlap with metadata cells in count matrix rows:\n")
print(overlap_rows)

if (overlap_rows > overlap_cols) {
  cat("[INFO] Counts matrix appears to be cells x genes. Transposing to genes x cells...\n")
  counts <- t(counts)
} else {
  cat("[INFO] Counts matrix appears to be genes x cells. No transpose needed.\n")
}

cat("[INFO] Count matrix after orientation check, genes x cells:\n")
print(dim(counts))



[INFO] Overlap with metadata cells in count matrix columns:
[1] 150849
[INFO] Overlap with metadata cells in count matrix rows:
[1] 0
[INFO] Counts matrix appears to be genes x cells. No transpose needed.
[INFO] Count matrix after orientation check, genes x cells:
[1]  19790 150849


In [7]:
# =========================================================
# 6. Align counts and metadata
# =========================================================

common_cells <- intersect(colnames(counts), rownames(metadata))

cat("[INFO] Number of common cells:\n")
print(length(common_cells))

if (length(common_cells) == 0) {
  stop("[ERROR] No common cell IDs between counts and metadata.")
}

counts <- counts[, common_cells, drop = FALSE]
metadata <- metadata[common_cells, , drop = FALSE]

cat("[INFO] Counts after alignment, genes x cells:\n")
print(dim(counts))

cat("[INFO] Metadata after alignment:\n")
print(dim(metadata))


# =========================================================
# 7. Save full expression matrix and cell labels
# =========================================================
# 注意：
# 全基因表达矩阵通常非常大，不建议直接写成 CSV。
# 这里保存为 RDS，避免内存和硬盘爆炸。

full_expr_gene_by_cell_path <- file.path(
  out_dir,
  "full_raw_counts_gene_by_cell.rds"
)

full_expr_cell_by_gene_path <- file.path(
  out_dir,
  "full_raw_counts_cell_by_gene.rds"
)

metadata_aligned_path <- file.path(
  out_dir,
  "metadata_aligned.csv"
)

cell_label_path <- file.path(
  out_dir,
  "cell_labels.csv"
)

saveRDS(
  counts,
  file = full_expr_gene_by_cell_path
)

saveRDS(
  t(counts),
  file = full_expr_cell_by_gene_path
)

metadata_out <- cbind(
  Cell = rownames(metadata),
  metadata
)

fwrite(
  metadata_out,
  file = metadata_aligned_path
)

cell_label_df <- data.frame(
  Cell = rownames(metadata),
  Label = metadata[[label_col]]
)

fwrite(
  cell_label_df,
  file = cell_label_path
)

cat("[OK] Full raw count matrix saved, genes x cells:\n")
cat(full_expr_gene_by_cell_path, "\n")

cat("[OK] Full raw count matrix saved, cells x genes:\n")
cat(full_expr_cell_by_gene_path, "\n")

cat("[OK] Metadata saved:\n")
cat(metadata_aligned_path, "\n")

cat("[OK] Cell labels saved:\n")
cat(cell_label_path, "\n")


# =========================================================
# 8. Create Seurat object and QC
# =========================================================
# min.cells = 3:
#   keep genes detected in at least 3 cells
#
# min.features = 200:
#   keep cells with at least 200 detected genes

seurat_obj <- CreateSeuratObject(
  counts = counts,
  project = "GSE179994_Tcell",
  meta.data = metadata,
  min.cells = 3,
  min.features = 200
)

cat("[INFO] Seurat object after QC:\n")
print(seurat_obj)

qc_summary <- data.frame(
  n_genes_before_qc = nrow(counts),
  n_cells_before_qc = ncol(counts),
  n_genes_after_qc = nrow(seurat_obj),
  n_cells_after_qc = ncol(seurat_obj)
)

fwrite(
  qc_summary,
  file = file.path(out_dir, "QC_summary.csv")
)



[INFO] Number of common cells:
[1] 150849
[INFO] Counts after alignment, genes x cells:
[1]  19790 150849
[INFO] Metadata after alignment:
[1] 150849      5
[OK] Full raw count matrix saved, genes x cells:
/home/zhanghang/GSE179994/Seurat_CellEnergy_full_pipeline/full_raw_counts_gene_by_cell.rds 
[OK] Full raw count matrix saved, cells x genes:
/home/zhanghang/GSE179994/Seurat_CellEnergy_full_pipeline/full_raw_counts_cell_by_gene.rds 
[OK] Metadata saved:
/home/zhanghang/GSE179994/Seurat_CellEnergy_full_pipeline/metadata_aligned.csv 
[OK] Cell labels saved:
/home/zhanghang/GSE179994/Seurat_CellEnergy_full_pipeline/cell_labels.csv 
[INFO] Seurat object after QC:
An object of class Seurat 
17277 features across 150849 samples within 1 assay 
Active assay: RNA (17277 features, 0 variable features)
 1 layer present: counts


In [8]:
# =========================================================
# 9. Normalize raw UMI counts
# =========================================================
# LogNormalize:
# log1p(count / total_counts_per_cell * 10000)
#
# No linear regression here.

seurat_obj <- NormalizeData(
  seurat_obj,
  normalization.method = "LogNormalize",
  scale.factor = 10000
)


# =========================================================
# 10. Identify highly variable genes
# =========================================================

seurat_obj <- FindVariableFeatures(
  seurat_obj,
  selection.method = "vst",
  nfeatures = 2000
)

hvg_genes <- VariableFeatures(seurat_obj)

cat("[INFO] Number of HVGs:\n")
print(length(hvg_genes))

hvg_path <- file.path(out_dir, "HVG_2000_gene_list.csv")

fwrite(
  data.frame(HVG = hvg_genes),
  file = hvg_path
)

cat("[OK] HVG list saved:\n")
cat(hvg_path, "\n")


# =========================================================
# 11. Scale HVG expression
# =========================================================
# This only performs centering and scaling.
# No vars.to.regress is used.
# Therefore, no linear regression is performed.

seurat_obj <- ScaleData(
  seurat_obj,
  features = hvg_genes
)


# =========================================================
# 12. Extract HVG matrices
# =========================================================

raw_hvg_counts_gene_by_cell <- get_assay_data_compat(
  seurat_obj,
  assay = "RNA",
  layer_name = "counts",
  slot_name = "counts"
)[hvg_genes, ]

normalized_hvg_gene_by_cell <- get_assay_data_compat(
  seurat_obj,
  assay = "RNA",
  layer_name = "data",
  slot_name = "data"
)[hvg_genes, ]

scaled_hvg_gene_by_cell <- get_assay_data_compat(
  seurat_obj,
  assay = "RNA",
  layer_name = "scale.data",
  slot_name = "scale.data"
)[hvg_genes, ]

cat("[INFO] Raw HVG counts, genes x cells:\n")
print(dim(raw_hvg_counts_gene_by_cell))

cat("[INFO] Normalized HVG expression, genes x cells:\n")
print(dim(normalized_hvg_gene_by_cell))

cat("[INFO] Scaled HVG expression, genes x cells:\n")
print(dim(scaled_hvg_gene_by_cell))


# =========================================================
# 13. Save HVG matrices
# =========================================================

raw_hvg_gene_by_cell_path <- file.path(
  out_dir,
  "raw_HVG_counts_gene_by_cell.csv"
)

normalized_hvg_gene_by_cell_path <- file.path(
  out_dir,
  "normalized_HVG_expr_gene_by_cell.csv"
)

scaled_hvg_gene_by_cell_path <- file.path(
  out_dir,
  "scaled_HVG_expr_gene_by_cell_for_CellEnergy.csv"
)

raw_hvg_cell_by_gene_path <- file.path(
  out_dir,
  "raw_HVG_counts_cell_by_gene.csv"
)

normalized_hvg_cell_by_gene_path <- file.path(
  out_dir,
  "normalized_HVG_expr_cell_by_gene.csv"
)

scaled_hvg_cell_by_gene_path <- file.path(
  out_dir,
  "scaled_HVG_expr_cell_by_gene.csv"
)

write_matrix_with_id(
  raw_hvg_counts_gene_by_cell,
  raw_hvg_gene_by_cell_path,
  id_col = "Gene"
)

write_matrix_with_id(
  normalized_hvg_gene_by_cell,
  normalized_hvg_gene_by_cell_path,
  id_col = "Gene"
)

# CellEnergy 输入文件：genes x cells，保留 row.names
write.csv(
  as.data.frame(as.matrix(scaled_hvg_gene_by_cell), check.names = FALSE),
  file = scaled_hvg_gene_by_cell_path,
  quote = FALSE,
  row.names = TRUE
)

write_matrix_with_id(
  t(as.matrix(raw_hvg_counts_gene_by_cell)),
  raw_hvg_cell_by_gene_path,
  id_col = "Cell"
)

write_matrix_with_id(
  t(as.matrix(normalized_hvg_gene_by_cell)),
  normalized_hvg_cell_by_gene_path,
  id_col = "Cell"
)

write_matrix_with_id(
  t(as.matrix(scaled_hvg_gene_by_cell)),
  scaled_hvg_cell_by_gene_path,
  id_col = "Cell"
)

cat("[OK] HVG matrices saved.\n")


Normalizing layer: counts

Finding variable features for layer counts



[INFO] Number of HVGs:
[1] 2000
[OK] HVG list saved:
/home/zhanghang/GSE179994/Seurat_CellEnergy_full_pipeline/HVG_2000_gene_list.csv 


Centering and scaling data matrix



[INFO] Raw HVG counts, genes x cells:
[1]   2000 150849
[INFO] Normalized HVG expression, genes x cells:
[1]   2000 150849
[INFO] Scaled HVG expression, genes x cells:
[1]   2000 150849


Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 2.2 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 2.2 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 2.2 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 2.2 GiB”


[OK] HVG matrices saved.


In [10]:
# =========================================================
# 14. Run CellEnergy / GLNE
# =========================================================
# 如果这里报：
# No module named 'pandas'
#
# 先在 R 里运行：
library(reticulate)
reticulate::py_install(c("pandas", "numpy", "scipy"), pip = TRUE)

# 然后重启 R，再重新跑。

cat("[INFO] Running CellEnergy calcGEn...\n")

result <- calcGEn(
  scaled_hvg_gene_by_cell_path,
  verbose = TRUE
)

scEn <- result$scEn
GLNE <- result$GLNE

cat("[INFO] scEnergy preview:\n")
print(head(scEn))

cat("[INFO] GLNE dimension before orientation check:\n")
print(dim(GLNE))


# =========================================================
# 15. Save CellEnergy outputs
# =========================================================

energy_path <- file.path(out_dir, "Energy_scEn.csv")
glne_gene_by_cell_path <- file.path(out_dir, "GLNE_gene_by_cell.csv")

# scEn 可能是 vector，也可能是 data.frame/matrix
scEn_df <- as.data.frame(scEn, check.names = FALSE)

if (is.null(rownames(scEn_df))) {
  rownames(scEn_df) <- colnames(scaled_hvg_gene_by_cell)
}

scEn_df <- cbind(
  Cell = rownames(scEn_df),
  scEn_df
)

fwrite(
  scEn_df,
  file = energy_path
)

write_matrix_with_id(
  GLNE,
  glne_gene_by_cell_path,
  id_col = "Gene"
)

cat("[OK] Single-cell energy scores saved:\n")
cat(energy_path, "\n")

cat("[OK] GLNE gene x cell matrix saved:\n")
cat(glne_gene_by_cell_path, "\n")


# =========================================================
# 16. Convert GLNE to cells x genes
# =========================================================

GLNE_mat <- as.matrix(GLNE)

raw_hvg_cell_by_gene <- t(as.matrix(raw_hvg_counts_gene_by_cell))
scaled_hvg_cell_by_gene <- t(as.matrix(scaled_hvg_gene_by_cell))

glne_overlap_cols_cells <- sum(colnames(GLNE_mat) %in% rownames(raw_hvg_cell_by_gene))
glne_overlap_rows_cells <- sum(rownames(GLNE_mat) %in% rownames(raw_hvg_cell_by_gene))

cat("[INFO] GLNE overlap with cells in columns:\n")
print(glne_overlap_cols_cells)

cat("[INFO] GLNE overlap with cells in rows:\n")
print(glne_overlap_rows_cells)

if (glne_overlap_rows_cells > glne_overlap_cols_cells) {
  cat("[INFO] GLNE appears to be cells x genes. No transpose needed.\n")
  GLNE_cell_by_gene <- GLNE_mat
} else {
  cat("[INFO] GLNE appears to be genes x cells. Transposing to cells x genes...\n")
  GLNE_cell_by_gene <- t(GLNE_mat)
}

common_cells_final <- intersect(rownames(raw_hvg_cell_by_gene), rownames(GLNE_cell_by_gene))
common_genes_final <- intersect(colnames(raw_hvg_cell_by_gene), colnames(GLNE_cell_by_gene))

cat("[INFO] Common cells between HVG expression and GLNE:\n")
print(length(common_cells_final))

cat("[INFO] Common genes between HVG expression and GLNE:\n")
print(length(common_genes_final))

if (length(common_cells_final) == 0) {
  stop("[ERROR] No common cells between HVG expression and GLNE.")
}

if (length(common_genes_final) == 0) {
  stop("[ERROR] No common genes between HVG expression and GLNE.")
}

raw_hvg_cell_by_gene <- raw_hvg_cell_by_gene[
  common_cells_final,
  common_genes_final,
  drop = FALSE
]

scaled_hvg_cell_by_gene <- scaled_hvg_cell_by_gene[
  common_cells_final,
  common_genes_final,
  drop = FALSE
]

GLNE_cell_by_gene <- GLNE_cell_by_gene[
  common_cells_final,
  common_genes_final,
  drop = FALSE
]

cat("[INFO] Final raw HVG matrix, cells x genes:\n")
print(dim(raw_hvg_cell_by_gene))

cat("[INFO] Final scaled HVG matrix, cells x genes:\n")
print(dim(scaled_hvg_cell_by_gene))

cat("[INFO] Final GLNE matrix, cells x genes:\n")
print(dim(GLNE_cell_by_gene))


# =========================================================
# 17. Save final cells x genes matrices
# =========================================================

final_raw_hvg_path <- file.path(
  out_dir,
  "final_raw_HVG_counts_cell_by_gene.csv"
)

final_scaled_hvg_path <- file.path(
  out_dir,
  "final_scaled_HVG_expr_cell_by_gene.csv"
)

final_glne_path <- file.path(
  out_dir,
  "final_GLNE_cell_by_gene.csv"
)

write_matrix_with_id(
  raw_hvg_cell_by_gene,
  final_raw_hvg_path,
  id_col = "Cell"
)

write_matrix_with_id(
  scaled_hvg_cell_by_gene,
  final_scaled_hvg_path,
  id_col = "Cell"
)

write_matrix_with_id(
  GLNE_cell_by_gene,
  final_glne_path,
  id_col = "Cell"
)

cat("[OK] Final raw HVG expression matrix saved:\n")
cat(final_raw_hvg_path, "\n")

cat("[OK] Final scaled HVG expression matrix saved:\n")
cat(final_scaled_hvg_path, "\n")

cat("[OK] Final GLNE matrix saved:\n")
cat(final_glne_path, "\n")


Warning message in reticulate::py_install(c("pandas", "numpy", "scipy"), pip = TRUE):
“An ephemeral virtual environment managed by 'reticulate' is currently in use.
To add more packages to your current session, call `py_require()` instead
of `py_install()`. Running:
  `py_require(c("pandas", "numpy", "scipy"))`”


[INFO] Running CellEnergy calcGEn...


2026-06-10 19:24:09.077253: Starting calculation.

2026-06-10 19:58:55.108069: Complete GLNE and cell Energy calculation.



[INFO] scEnergy preview:
                         scEnergy  scEn_Nor
P1.ut.AAACCTGAGGCACATG-1   -19900 0.9196885
P1.ut.AAACCTGGTGGTACAG-1   -23436 0.9048048
P1.ut.AAACCTGTCACGGTTA-1   -38781 0.8402147
P1.ut.AAACCTGTCCGCTGTT-1   -99681 0.5838746
P1.ut.AAACCTGTCTGGTATG-1   -18336 0.9262717
P1.ut.AAACCTGTCTTGAGAC-1   -13530 0.9465011
[INFO] GLNE dimension before orientation check:
[1]   2000 150849
[OK] Single-cell energy scores saved:
/home/zhanghang/GSE179994/Seurat_CellEnergy_full_pipeline/Energy_scEn.csv 
[OK] GLNE gene x cell matrix saved:
/home/zhanghang/GSE179994/Seurat_CellEnergy_full_pipeline/GLNE_gene_by_cell.csv 


Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 2.2 GiB”


[INFO] GLNE overlap with cells in columns:
[1] 150849
[INFO] GLNE overlap with cells in rows:
[1] 0
[INFO] GLNE appears to be genes x cells. Transposing to cells x genes...
[INFO] Common cells between HVG expression and GLNE:
[1] 150849
[INFO] Common genes between HVG expression and GLNE:
[1] 2000
[INFO] Final raw HVG matrix, cells x genes:
[1] 150849   2000
[INFO] Final scaled HVG matrix, cells x genes:
[1] 150849   2000
[INFO] Final GLNE matrix, cells x genes:
[1] 150849   2000
[OK] Final raw HVG expression matrix saved:
/home/zhanghang/GSE179994/Seurat_CellEnergy_full_pipeline/final_raw_HVG_counts_cell_by_gene.csv 
[OK] Final scaled HVG expression matrix saved:
/home/zhanghang/GSE179994/Seurat_CellEnergy_full_pipeline/final_scaled_HVG_expr_cell_by_gene.csv 
[OK] Final GLNE matrix saved:
/home/zhanghang/GSE179994/Seurat_CellEnergy_full_pipeline/final_GLNE_cell_by_gene.csv 


In [11]:
# =========================================================
# 18. MinMax normalization for downstream learning
# =========================================================
# Input:
# rows = cells
# columns = genes/features
#
# View 1:
# raw HVG count matrix
#
# View 2:
# GLNE energy matrix

raw_hvg_minmax <- minmax_scale_columns(raw_hvg_cell_by_gene)
GLNE_minmax <- minmax_scale_columns(GLNE_cell_by_gene)

raw_hvg_minmax_path <- file.path(
  out_dir,
  "final_raw_HVG_counts_MinMax_cell_by_gene.csv"
)

GLNE_minmax_path <- file.path(
  out_dir,
  "final_GLNE_MinMax_cell_by_gene.csv"
)

write_matrix_with_id(
  raw_hvg_minmax,
  raw_hvg_minmax_path,
  id_col = "Cell"
)

write_matrix_with_id(
  GLNE_minmax,
  GLNE_minmax_path,
  id_col = "Cell"
)

cat("[OK] Final MinMax raw HVG matrix saved:\n")
cat(raw_hvg_minmax_path, "\n")

cat("[OK] Final MinMax GLNE matrix saved:\n")
cat(GLNE_minmax_path, "\n")


# =========================================================
# 19. Save final metadata and Seurat object
# =========================================================

metadata_final <- seurat_obj@meta.data
metadata_final <- metadata_final[common_cells_final, , drop = FALSE]

metadata_final_path <- file.path(
  out_dir,
  "metadata_final_aligned_after_QC.csv"
)

metadata_final_out <- cbind(
  Cell = rownames(metadata_final),
  metadata_final
)

fwrite(
  metadata_final_out,
  file = metadata_final_path
)

seurat_rds_path <- file.path(
  out_dir,
  "GSE179994_Tcell_Seurat_CellEnergy_processed_no_regression.rds"
)

saveRDS(
  seurat_obj,
  file = seurat_rds_path
)

cat("[OK] Final metadata saved:\n")
cat(metadata_final_path, "\n")

cat("[OK] Seurat object saved:\n")
cat(seurat_rds_path, "\n")


# =========================================================
# 20. Save summary
# =========================================================

summary_df <- data.frame(
  item = c(
    "raw_genes_before_qc",
    "raw_cells_before_qc",
    "genes_after_qc",
    "cells_after_qc",
    "hvg_number",
    "final_common_cells",
    "final_common_genes"
  ),
  value = c(
    nrow(counts),
    ncol(counts),
    nrow(seurat_obj),
    ncol(seurat_obj),
    length(hvg_genes),
    nrow(raw_hvg_cell_by_gene),
    ncol(raw_hvg_cell_by_gene)
  )
)

summary_path <- file.path(
  out_dir,
  "pipeline_summary.csv"
)

fwrite(
  summary_df,
  file = summary_path
)

cat("[OK] Pipeline summary saved:\n")
cat(summary_path, "\n")


# =========================================================
# 21. Done
# =========================================================

cat("\n[DONE] Full pipeline finished successfully.\n")
cat("[DONE] No linear regression was performed.\n\n")

cat("Main outputs:\n")
cat("1. Full raw expression matrix, genes x cells:\n")
cat(full_expr_gene_by_cell_path, "\n\n")

cat("2. Full raw expression matrix, cells x genes:\n")
cat(full_expr_cell_by_gene_path, "\n\n")

cat("3. Cell labels:\n")
cat(cell_label_path, "\n\n")

cat("4. HVG gene list:\n")
cat(hvg_path, "\n\n")

cat("5. Final raw HVG expression, cells x genes:\n")
cat(final_raw_hvg_path, "\n\n")

cat("6. Final scaled HVG expression, cells x genes:\n")
cat(final_scaled_hvg_path, "\n\n")

cat("7. Final GLNE energy matrix, cells x genes:\n")
cat(final_glne_path, "\n\n")

cat("8. View 1 for downstream model:\n")
cat(raw_hvg_minmax_path, "\n\n")

cat("9. View 2 for downstream model:\n")
cat(GLNE_minmax_path, "\n\n")

cat("10. Final aligned metadata:\n")
cat(metadata_final_path, "\n")

[OK] Final MinMax raw HVG matrix saved:
/home/zhanghang/GSE179994/Seurat_CellEnergy_full_pipeline/final_raw_HVG_counts_MinMax_cell_by_gene.csv 
[OK] Final MinMax GLNE matrix saved:
/home/zhanghang/GSE179994/Seurat_CellEnergy_full_pipeline/final_GLNE_MinMax_cell_by_gene.csv 
[OK] Final metadata saved:
/home/zhanghang/GSE179994/Seurat_CellEnergy_full_pipeline/metadata_final_aligned_after_QC.csv 
[OK] Seurat object saved:
/home/zhanghang/GSE179994/Seurat_CellEnergy_full_pipeline/GSE179994_Tcell_Seurat_CellEnergy_processed_no_regression.rds 
[OK] Pipeline summary saved:
/home/zhanghang/GSE179994/Seurat_CellEnergy_full_pipeline/pipeline_summary.csv 

[DONE] Full pipeline finished successfully.
[DONE] No linear regression was performed.

Main outputs:
1. Full raw expression matrix, genes x cells:
/home/zhanghang/GSE179994/Seurat_CellEnergy_full_pipeline/full_raw_counts_gene_by_cell.rds 

2. Full raw expression matrix, cells x genes:
/home/zhanghang/GSE179994/Seurat_CellEnergy_full_pipeline/f